# Setup

In [ ]:
suggestive_t=1e-5
gw_t = 0.05/334605 
apoe_str_id = 'chr19:44909968:GGGGGGGGGGG:GGGGGGGGG'
apoe_snp_id = '19_45411941_rs429358_T_C'
APOE_POS=44908684

In [ ]:
# code from other analyses I will need probably
## imports
import sys
import os

from scipy import stats

import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

# plt.rcParams.update({'font.size': 22})
font_kwargs={
    'fontsize':18
}
sns.set_theme(font_scale=1.2, context='paper', style='white', palette='tab10')
# set default stuff
plink_keys=['#CHROM', "POS"]
meta_keys=['CHR', "BP"]

scatter_palette={
    'SNP':'tab:blue', 
    'STR':'tab:orange', 
    'adjusted':'tomato', 
    'unadjusted':'teal'
}
custom_palette = sns.color_palette("tab10", 4)
# manhatti_palette = sns.color_palette()[:4]
manhatti_palette = {}
for c in range(1, 23):
    manhatti_palette[c] = custom_palette[c % len(custom_palette)]

# methods
def get_results(file_path, is_meta=False, cut=True, cut_threshold=10):
    df = pd.read_csv(file_path, sep="\t")
    initial = df.shape[0]
    df = df.dropna(axis=0)
    after = df.shape[0]
    print(f'drop {initial-after} rows because of NA values')
    df = df.astype({'P':float})
    df.loc[df.P<sys.float_info.min, 'P']=sys.float_info.min

    min_ct = df[df.P == sys.float_info.min].shape[0]
    if min_ct > 0:
        print(f'set {min_ct} P values to maximum possible min of {sys.float_info.min}')
    df['-log10'] = -np.log10(df.P)

    if not is_meta:
        df = df[df['TEST']=='ADD']

    if cut:
        df=df[df['-log10'] < cut_threshold]
    
    return df

# %% Plotting
def manhatti(df, title, sort_keys=['#CHROM', "POS"], hue_key='#CHROM', style_key=None, y_steps=10, legend=None, 
             sort_key='-log10', sort_asc=True, plot_y_thresh=True, do_full=True, suggest_t=-np.log10(suggestive_t), p_sig_t=-np.log10(gw_t), palette=manhatti_palette):
    plot_df = df.copy(deep=True)
    plt.figure(figsize=(30,10))
    if do_full:
        chr_maxxs = plot_df.groupby('#CHROM')['POS'].max()
        chroms_sorted = sorted(chr_maxxs.keys())
        chr_starts = {chroms_sorted[0]: 0}
        for idx, c in enumerate(chroms_sorted[1:], start=1):
            prev_c = chroms_sorted[idx - 1]
            chr_starts[c] = chr_starts[prev_c] + chr_maxxs[prev_c] + 1
        plot_df['i'] = [p + chr_starts[c] for p,c in zip(plot_df['POS'], plot_df['#CHROM'])]
    else:
        plot_df = df.sort_values(['numeric_chr', sort_keys[1]])
        plot_df = plot_df.reset_index(drop=True); 
        plot_df['i'] = plot_df.index

    plot_df = plot_df.sort_values(by=sort_key, ascending=sort_asc)
    plot = sns.scatterplot(plot_df, x='i', y='-log10', hue=hue_key, legend=legend, linewidth=0.3, style=style_key, palette=palette)
    labels_df=plot_df.groupby(sort_keys[0])['i'].median()
    plot.set_xlabel('chr')
    plot.set_xticks(labels_df)
    plot.set_xticklabels(labels_df.index)
    max_x = plot_df.i.max()
    max_x = max_x + 0.01*max_x
    min_x = plot_df.i.min() - 0.01*max_x
    plt.xlim([min_x, max_x])

    max_y=int(max(plot_df['-log10'].max(), y_steps))+1
    step = int(max_y/y_steps)
    
    plot.set_yticks(range(1, max_y, step))
    if plot_y_thresh:
        plt.hlines(y=[p_sig_t, suggest_t], xmin=min_x, xmax=max_x, colors=['tab:red', 'k'], linestyles='dashed')

    plot.grid(axis='y')
    plot.set_title(title)
    plt.grid(axis='y')
    plt.title(title)
    plt.tight_layout()

    return plot.figure

In [ ]:
def altered_broken_manhatti_multi(df, title, sort_keys=['#CHROM', "POS"], hue_key='#CHROM', style_key=None, legend=None, 
             sort_key='-log10', sort_asc=True, plot_y_thresh=True, do_full=True, suggest_t=5, p_sig_t=-np.log10(gw_t), plot_break=(30,100), kwargs={}, bi_key='biallelic approach'):
    plot_df = df.copy(deep=True)
    f, (outlier_ax, plot_ax) = plt.subplots(2, 1, sharex=True, figsize=(30,10), height_ratios=[1,8])
    if do_full:
        chr_maxxs = plot_df.groupby('#CHROM')['POS'].max()
        chroms_sorted = sorted(chr_maxxs.keys())
        chr_starts = {chroms_sorted[0]: 0}
        for idx, c in enumerate(chroms_sorted[1:], start=1):
            prev_c = chroms_sorted[idx - 1]
            chr_starts[c] = chr_starts[prev_c] + chr_maxxs[prev_c] + 1
        plot_df['i'] = [p + chr_starts[c] for p,c in zip(plot_df['POS'], plot_df['#CHROM'])]
    else:
        plot_df = df.sort_values(sort_keys)
        plot_df = plot_df.reset_index(drop=True); 
        plot_df['i'] = plot_df.index

    plot_df = plot_df.sort_values(by=[sort_key, '-log10'], ascending=[sort_asc, True])
    mask = (plot_df['#CHROM'] % 2 == 0) & (plot_df['source'] == bi_key)
    plot = sns.scatterplot(plot_df[mask], x='i', y='-log10', color='cornflowerblue', legend=None, linewidth=0.3, style=style_key, ax=plot_ax, **kwargs)

    mask = (plot_df['#CHROM'] % 2 == 1) | (plot_df['source'] != bi_key)
    plot = sns.scatterplot(plot_df[mask], x='i', y='-log10', hue=hue_key, palette=manhatti_palette, legend=None, linewidth=0.3, style=style_key, ax=plot_ax, **kwargs)

    labels_df=plot_df.groupby(sort_keys[0])['i'].agg(['median', 'min', 'max'])
    
    labels_df['mid'] = (labels_df['max'].shift() + labels_df['max']) / 2
    labels_df.loc[labels_df['mid'].isna(), 'mid'] = labels_df['max'].iloc[0] / 2

    plot_ax.set_xticks(labels_df['max'])
    plot_ax.set_xticklabels([], minor=False)

    plot_ax.set_xticks(labels_df['mid'], minor=True)
    plot_ax.set_xticklabels(labels_df.index, minor=True)
    plot_ax.xaxis.grid(True)
    
    outlier_ax.set_xticks(labels_df['max'])
    outlier_ax.set_xticklabels([], minor=False)

    outlier_ax.set_xticks(labels_df['mid'], minor=True)
    outlier_ax.set_xticklabels(labels_df.index, minor=True)
    outlier_ax.xaxis.grid(True)

    max_x = plot_df.i.max()
    max_x = max_x + 0.01*max_x
    min_x = plot_df.i.min() - 0.01*max_x
    plt.xlim([min_x, max_x])
    plot_ax.set_yticks(range(0, 30, 5))

    plot.set_xlabel('Chromosome', fontdict=font_kwargs)
    plot.set_ylabel('-log10(p)', fontdict=font_kwargs)

    if plot_y_thresh:
        plt.hlines(y=[p_sig_t, suggest_t], xmin=min_x, xmax=max_x, colors=['tab:red', 'k'], linestyles='dashed')
    plt.grid(axis='y')
        # plt.tight_layout()

    sns.scatterplot(plot_df, x='i', y='-log10', hue=hue_key, palette=manhatti_palette, legend=legend, linewidth=0.3, style=style_key, ax=outlier_ax)
    y_max_outliers = outlier_ax.get_ylim()[1]
    y_shift = y_max_outliers + y_max_outliers*0.1
    outlier_ax.set_ylim(bottom=plot_break[1], top=y_shift)  # outliers only
    plot_ax.set_ylim(0, plot_break[0])  # most of the data

    outlier_ax.spines["bottom"].set_visible(False)
    plot_ax.spines["top"].set_visible(False)

    # outlier_ax.xaxis.tick_top()
    outlier_ax.tick_params(top=False, bottom=False)

    d = .25  # proportion of vertical to horizontal extent of the slanted line
    ax_kwargs = dict(marker=[(-1, -d), (1, d)], markersize=12,
                linestyle="none", color='k', mec='k', mew=1, clip_on=False)
    outlier_ax.plot([0, 1], [0, 0], transform=outlier_ax.transAxes, **ax_kwargs)
    outlier_ax.set_ylabel('')
    plot_ax.plot([0, 1], [1, 1], transform=plot_ax.transAxes, **ax_kwargs)



    plt.subplots_adjust(hspace=0.05)
    outlier_ax.set_title(title, **font_kwargs)
    plt.tight_layout()

    return plot.figure

In [ ]:
# %% Code for Miami plots, created by Gemini based on the initial manhatti function and further modified
def calculate_coords(df_top, df_bot, do_full=True, sort_keys=['#CHROM', "POS"]):
    """
    Helper to calculate cumulative coordinates consistent across both DFs.
    """
    # Combine only necessary columns to find global max per chromosome
    combined = pd.concat([df_top[['#CHROM', 'POS']], df_bot[['#CHROM', 'POS']]])
    
    df_top = df_top.copy()
    df_bot = df_bot.copy()
    
    if do_full:
        # Calculate max position per chromosome across BOTH datasets
        chr_maxxs = combined.groupby('#CHROM')['POS'].max()
        chroms_sorted = sorted(chr_maxxs.keys())
        
        # Calculate cumulative offsets
        chr_starts = {chroms_sorted[0]: 0}
        for idx, c in enumerate(chroms_sorted[1:], start=1):
            prev_c = chroms_sorted[idx - 1]
            chr_starts[c] = chr_starts[prev_c] + chr_maxxs[prev_c] + 1
            
        # Apply to both DFs
        df_top['i'] = [p + chr_starts[c] for p, c in zip(df_top['POS'], df_top['#CHROM'])]
        df_bot['i'] = [p + chr_starts[c] for p, c in zip(df_bot['POS'], df_bot['#CHROM'])]
    else:
        # Simple sorting if not using genomic coordinates
        df_top = df_top.sort_values(sort_keys).reset_index(drop=True)
        df_bot = df_bot.sort_values(sort_keys).reset_index(drop=True)
        df_top['i'] = df_top.index
        df_bot['i'] = df_bot.index

    return df_top, df_bot


def miami_plot(df_top, df_bot, title, label_top="Dataset 1", label_bot="Dataset 2",
               sort_keys=['#CHROM', "POS"], hue_key='#CHROM', style_key=None, y_steps=10, 
               legend=None, sort_key='-log10', sort_asc=True, plot_y_thresh=True, 
               do_full=True, suggest_t=5, p_sig_t=7.3, palette=None):

    # 1. Calculate unified coordinates
    plot_df_top, plot_df_bot = calculate_coords(df_top, df_bot, do_full, sort_keys)
    
    # Sort by p-value for plotting order (so significant points represent on top)
    plot_df_top = plot_df_top.sort_values(by=sort_key, ascending=sort_asc)
    plot_df_bot = plot_df_bot.sort_values(by=sort_key, ascending=sort_asc)

    # 2. Setup Subplots (2 rows, shared X)
    fig, (ax_top, ax_bot) = plt.subplots(2, 1, sharex=True, figsize=(30, 15))
    plt.subplots_adjust(hspace=0.065) # Remove space between plots

    # 3. Plotting
    # Top Plot
    sns.scatterplot(data=plot_df_top, x='i', y='-log10', hue=hue_key, legend=legend, 
                    linewidth=0.3, style=style_key, palette=palette, ax=ax_top)
    
    # Bottom Plot
    sns.scatterplot(data=plot_df_bot, x='i', y='-log10', hue=hue_key, legend=legend, 
                    linewidth=0.3, style=style_key, palette=palette, ax=ax_bot)

    # 4. Axis Manipulation
    ax_bot.invert_yaxis() # This creates the Miami effect
    
    # X-Axis Labels (Only needed on bottom, calculated from Top DF or combined)
    labels_df = plot_df_top.groupby(sort_keys[0])['i'].median()
    
    # Set Bottom X-axis
    ax_bot.set_xlabel('')
    ax_bot.tick_params(axis='x', which='both', bottom=False, top=True, labelbottom=False)

    # Clean up Top X-axis
    ax_top.set_xlabel('Chromosome')
    ax_top.set_xticklabels(labels_df.index)
    ax_top.set_xticks(labels_df)
    ax_top.tick_params(axis='x', which='both', bottom=True, top=False, labelbottom=True)
    

    # Calculate X limits
    max_x = max(plot_df_top.i.max(), plot_df_bot.i.max())
    max_x = max_x + 0.01 * max_x
    min_x = min(plot_df_top.i.min(), plot_df_bot.i.min())
    plt.xlim([min_x, max_x])

    # 5. Y-Axis and Thresholds
    # Determine max Y to keep scales symmetrical (optional, but looks better)
    max_y_val = max(plot_df_top['-log10'].max(), plot_df_bot['-log10'].max(), y_steps) + 1
    step = int(max_y_val / y_steps) if int(max_y_val / y_steps) > 0 else 1
    
    ticks = range(0, int(max_y_val) + step, step)
    
    for ax in [ax_top, ax_bot]:
        ax.set_yticks(ticks)
        ax.set_ylabel(f'-log10(p)')
        ax.grid(axis='y', linestyle='--', alpha=0.5)
        
        if plot_y_thresh:
            ax.hlines(y=[p_sig_t, suggest_t], xmin=min_x, xmax=max_x, 
                      colors=['tab:red', 'k'], linestyles='dashed')

    # Titles and Legends
    ax_top.set_title(f"{title} (Top: {label_top} | Bottom: {label_bot})")
    plt.tight_layout()
    
    return fig

In [ ]:
def check_mkdir(dir_path):
    if not os.path.isdir(dir_path):
        os.makedirs(dir_path)

In [ ]:
def plot_corrs(df, x='OR_t0', y='OR_t1', title='', verbose=True):
    sns.scatterplot(df, x=x, y=y)
    # df_cp = df.dropna(axis=0).copy(deep=True)
    df_cp = df.replace(np.nan, 0).copy(deep=True)
    stat_out = stats.pearsonr(df_cp[x],df_cp[y])
    # plt.text(0.05, 0.95, 'Pearsons\'s', transform=plt.gca().transAxes)
    figtext = 'Pearson\'s rho={:.4f}'.format(stat_out.statistic) #+ ';  p={:.2e}'.format(stat_out.pvalue)
    plt.text(0.05, 0.95, figtext, transform=plt.gca().transAxes)
    plt.title(title)
    if verbose:
        print(stat_out)

In [ ]:
results_path='/data/str_gwa/data/eval/sex_strat'
check_mkdir(results_path)

In [ ]:
# load inital result for compariosn
initial_str_df = pd.read_csv(
    f'{results_path}/../initial_str_results_biallelic.csv', sep='\t'
    ).sort_values(by=['#CHROM', 'POS'])
print(f'Initial study STR results: n={len(initial_str_df)}')
initial_str_df.head()

In [ ]:
plot_run = True

In [ ]:
cols_of_interest = ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1', 'P', '-log10', 'BETA', 'SE', 'islocmax']

# Load sex stratified

## Overview

In [ ]:
females_df = get_results(
    f'{results_path}/str_biallelic.females.glm.linear' , cut=False
).sort_values(by=['#CHROM', 'POS'])
print(f'STR results, females: n={len(females_df)}')
females_df.head()

In [ ]:
males_df = get_results(
    f'{results_path}/str_biallelic.males.glm.linear' , cut=False
).sort_values(by=['#CHROM', 'POS'])
print(f'STR results, males: n={len(males_df)}')
males_df.head()

In [ ]:
if plot_run:
    mask = (females_df.P < 0.05) & (females_df.P > 1e-30)
    p = manhatti(
        females_df[mask], "STR GWAS results for female samples"
    )
    p.savefig(f'{results_path}/females_str_manhatti.png', dpi=300)
    

In [ ]:
if plot_run:
    mask = (males_df.P < 0.05) & (males_df.P > 1e-30)
    p = manhatti(
        males_df[mask], "STR GWAS results for male samples"
    )
    p.savefig(f'{results_path}/males_str_manhatti.png', dpi=300)

In [ ]:
if plot_run:
    mask_m = (males_df.P < 0.025) & (males_df.P > 1e-30)
    mask_f = (females_df.P < 0.025) & (females_df.P > 1e-30)
    p = miami_plot(males_df[mask_m], females_df[mask_f], "Miami plot of STR GWAS results stratified by sex", hue_key='#CHROM', palette=manhatti_palette, 
                    label_top='Male samples', label_bot='Female samples')
    p.savefig(f'{results_path}/miami_str_manhatti.png', dpi=300)

## Variable assessments

In [ ]:
male_sigs = males_df[males_df.P < suggestive_t]
female_sigs = females_df[females_df.P < suggestive_t]
overlap_sigs = set(male_sigs.ID).intersection(set(female_sigs.ID))
print(f'Number of suggestive signals in males: {len(male_sigs)}')
print(f'Number of suggestive signals in females: {len(female_sigs)}')
print(f'Number of overlapping suggestive signals: {len(overlap_sigs)}')

In [ ]:
matched = pd.merge(male_sigs, female_sigs, on=['ID', '#CHROM', 'POS', 'REF', 'ALT', 'A1'], suffixes=('_male', '_female'), how='outer').replace(np.nan, 'miss')
matched.head()

In [ ]:
matched[['-log10_male', '-log10_female']] = matched[['-log10_male', '-log10_female']].replace('miss', 0.0)
matched.head()

## Compare peaks

In [ ]:
def get_maxes_idxs(df, winsize=250000):
    keep=[]

    for i, r in df.iterrows():
        chr = r['#CHROM']
        pos = r.POS
        cands = df[(df['#CHROM'] == chr) 
                & (abs(df.POS-pos)<winsize)]
        min_idx = cands.P.idxmin()
        if min_idx == i: 
            keep.append(i)

    return keep

In [ ]:
male_lead_idxs = get_maxes_idxs(males_df[males_df.P < suggestive_t])
female_lead_idxs = get_maxes_idxs(females_df[females_df.P < suggestive_t])
males_df['islocmax'] = np.where(males_df.index.isin(male_lead_idxs), 1, 0)
females_df['islocmax'] = np.where(females_df.index.isin(female_lead_idxs), 1, 0)
lead_ids = set(males_df.loc[male_lead_idxs, 'ID'].tolist() + females_df.loc[female_lead_idxs, 'ID'].tolist())
print(f'Number of lead STRs (within 250kb windows) with p<{suggestive_t}: {len(lead_ids)}')

In [ ]:
lead_ids.update(initial_str_df[initial_str_df.islocmax == 1].ID.tolist())
initial_str_df[(initial_str_df.ID.isin(lead_ids))][
    cols_of_interest
]

In [ ]:

m_map = {x: f'{x}_male' for x in cols_of_interest if x not in ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1']}
m_map

In [ ]:
combined_leads = pd.merge(
    initial_str_df[initial_str_df.ID.isin(lead_ids)][cols_of_interest],
    females_df[females_df.ID.isin(lead_ids)][cols_of_interest],
    on=['#CHROM', 'POS', 'ID'], suffixes=('_initial', '_female')
)
combined_leads = pd.merge(
    combined_leads, 
    males_df[males_df.ID.isin(lead_ids)][cols_of_interest],
    on=['#CHROM', 'POS', 'ID'], suffixes=('', '_male')
).rename(columns=m_map)
print(f'Combined lead STRs from all analyses: n={len(combined_leads)}')
combined_leads.head()

In [ ]:
mask = (combined_leads.P_initial < gw_t) | (combined_leads.P_female < gw_t) | (combined_leads.P_male < gw_t)
print(mask.sum())
combined_leads = combined_leads[mask]
combined_leads = combined_leads.sort_values(by=['#CHROM', 'POS']).reset_index(drop=True)

In [ ]:
# Check for vars that are gw_sig in either study but do not show in the combined
miss_df = pd.DataFrame(columns=combined_leads.columns)
loc_maxs = combined_leads['islocmax_initial'] == 1
mask = (combined_leads.islocmax_male == 1) | (combined_leads.islocmax_female == 1)
print(f'Number of lead STRs that are locmax in the sex-stratified studies: {mask.sum()}')
for c,r in combined_leads[mask].iterrows():
    str_id = r.ID
    chr = r['#CHROM']
    pos = r.POS

    near_maxs = combined_leads[loc_maxs & (combined_leads['#CHROM'] == chr) & (abs(combined_leads.POS - pos) < 0.5e6)]
    if near_maxs.shape[0] == 0:
        print(f'No locmax STRs on chr{chr} for {str_id}')
        miss_df = pd.concat([miss_df, pd.DataFrame([r])], ignore_index=True)
        # print(r[cols_of_interest].to_markdown())
        continue

miss_df[
    ['ID', 'P_initial', 'P_male', 'P_female', '-log10_initial', '-log10_male', '-log10_female', 'islocmax_initial', 'islocmax_male', 'islocmax_female']
]


In [ ]:
# Show next best STRs in the initial study within 250kb of the missing lead STRs
next_best = pd.DataFrame(columns=combined_leads.columns)
for c,r in miss_df.iterrows():
    str_id = r.ID
    chr = r['#CHROM']
    pos = r.POS

    next_best = pd.concat([
        next_best, 
        initial_str_df[
            (initial_str_df['#CHROM'] == chr) & 
            (abs(initial_str_df.POS - pos) < 250000)
        ].sort_values(by='P').head(1)
    ])

next_best

# Sex Interaction

In [ ]:
# Eval Sex-interaction term
sex_interact_df = get_results(
    f'{results_path}/str_biallelic.sex_interact.glm.linear' , cut=False, is_meta=True
).sort_values(by=['#CHROM', 'POS'])
if plot_run:
    mask = (sex_interact_df.P < 0.05) & (sex_interact_df.P > 1e-30)
    p = manhatti(sex_interact_df[mask], "STR GWAS results for sex interaction term", hue_key='TEST', palette={'ADD':'teal', 'ADDxsex':'orange'}, style_key='TEST', legend='full')
# reduce to interaction test only
sex_interact_df = sex_interact_df[sex_interact_df['TEST'] == 'ADDxsex']
print(f'STR results, sex interaction: n={len(sex_interact_df)}')   
sex_interact_df.head()

In [ ]:
 # get lead vars for suggestive signals
idx = get_maxes_idxs(sex_interact_df[sex_interact_df.P < suggestive_t])
sex_interact_df['islocmax'] = 0
sex_interact_df.loc[idx, 'islocmax'] = 1
sex_interact_df.islocmax.value_counts()

In [ ]:
# plot interaction manhattan 
if plot_run:
    mask = (sex_interact_df.P < 0.05) & (sex_interact_df.P > 1e-30)
    mask = mask & (sex_interact_df.TEST == 'ADDxsex')
    p = manhatti(sex_interact_df[mask], "STR GWAS results for sex interaction term")
    p.savefig(f'{results_path}/sex_interaction_str_manhatti.png', dpi=300)

## Find lead signals in interaction results

In [ ]:
max_ids = sex_interact_df.loc[sex_interact_df['islocmax'] == 1, 'ID'].tolist()
sex_interact_df.loc[sex_interact_df.ID.isin(max_ids), ['#CHROM', 'POS', 'ID', 'P', '-log10', 'BETA']].sort_values(by=['#CHROM', 'POS']).reset_index(drop=True)

In [ ]:
# Show corresponding results in initial study
initial_str_df[initial_str_df.ID.isin(max_ids)][
    cols_of_interest
].sort_values(by=['#CHROM', 'POS']).reset_index(drop=True)

## Combine all

In [ ]:
# Set interest columns
combined_cols = ['#CHROM', 'POS', 'ID', 'P_initial', 'P_male', 'P_female', 'P_sex_interact', 'BETA_male', 'BETA_female', 'BETA_sex_interact', 'islocmax_initial', 'islocmax_male', 'islocmax_female', 'islocmax_sex_interact']
print('Combining all lead STRs from all analyses...')
print(f'Starting with {len(initial_str_df)} STRs from initial study')
combined_interact = pd.merge(
    initial_str_df[cols_of_interest],
    females_df[cols_of_interest],
    on=['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1'], suffixes=('_initial', '_female')
)
combined_interact = pd.merge(
    combined_interact, 
    males_df[cols_of_interest],
    on=['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1'], suffixes=('', '_male')
).rename(columns={x: f'{x}_male' for x in cols_of_interest if x not in ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1']})
print(f'Combined lead STRs from all analyses: n={len(combined_interact)}')

combined_interact = pd.merge(
    combined_interact,
    sex_interact_df[cols_of_interest],
    on=['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1']
).rename(columns={x: f'{x}_sex_interact' for x in cols_of_interest if x not in ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1']})
print(f'Combined lead STRs from all analyses: n={len(combined_interact)}')
combined_interact.head()

### Show interaction results for gw loci in initial study

In [ ]:
mask = (combined_interact.islocmax_initial == 1) & (combined_interact.P_initial < gw_t)
combined_interact[mask][
    ['#CHROM', 'POS', 'ID', 'P_initial', 'P_male', 'P_female', 'P_sex_interact', 'BETA_sex_interact', '-log10_sex_interact', 'islocmax_male', 'islocmax_female']
].sort_values(by=['#CHROM', 'POS']).reset_index(drop=True)

### Search for variants near interaction leads

In [ ]:
mask = combined_interact.islocmax_sex_interact == 1
combined_interact[mask][
    combined_cols
].sort_values(by=['#CHROM', 'POS']).reset_index(drop=True)

In [ ]:
# check for signals that are suggestive in interaction and one sex-specific analysis
mask = (combined_interact.P_sex_interact < suggestive_t) & ((combined_interact.P_male < suggestive_t) | (combined_interact.P_female < suggestive_t))
combined_interact[mask][
    combined_cols
].sort_values(by=['#CHROM', 'POS']).reset_index(drop=True).sort_values(by='P_male')

In [ ]:
# Are there any signals in the initial study nearby
mask = (initial_str_df['#CHROM'] == 17) & (abs(initial_str_df.POS - 74001907) < 250000)
initial_str_df[mask].sort_values(by='P').head()

In [ ]:
mask = (combined_interact['#CHROM'] == 17) & (abs(combined_interact.POS - 74001907) < 250000) & (combined_interact.P_male < 5e-3)
combined_interact[mask][
    combined_cols
].sort_values(by='P_male').reset_index(drop=True)

In [ ]:
# check for suport
mask = (combined_interact['#CHROM'] == 17) & (abs(combined_interact.POS - 74001907) < 5e5)
sns.scatterplot(combined_interact[mask], x='POS',  y='-log10_male')

In [ ]:
# same for the signal on chr9 -> does not show at least suggetive in any sex-specific analysis
mask = (combined_interact['#CHROM'] == 9) & (combined_interact.islocmax_sex_interact == 1)
combined_interact[mask][
    combined_cols
].sort_values(by=['#CHROM', 'POS']).reset_index(drop=True)

# Check individual maxes again

In [ ]:
max_male_ids = males_df.loc[males_df.islocmax == 1, 'ID'].tolist()
max_female_ids = females_df.loc[females_df.islocmax == 1, 'ID'].tolist()
max_sex_interact_ids = sex_interact_df.loc[sex_interact_df.islocmax == 1, 'ID'].tolist()
lead_ids_all = set(
    max_male_ids +  max_female_ids + max_sex_interact_ids +
    initial_str_df[initial_str_df.islocmax == 1].ID.tolist()
)
print(f'Individual leads counts: males={len(max_male_ids)}, females={len(max_female_ids)}, sex interaction={len(max_sex_interact_ids)}, initial study locmax={len(initial_str_df[initial_str_df.islocmax == 1])}')
print(f'Number of lead STRs (within 250kb windows) with p<{suggestive_t} across all analyses: {len(lead_ids_all)}') 

In [ ]:
combined_interact[combined_interact.ID.isin(max_sex_interact_ids)][
    combined_cols
].reset_index(drop=True)

In [ ]:
# Check female only < gw_t
combined_interact[(combined_interact.P_female < gw_t) & (combined_interact.P_male > gw_t) & ((combined_interact['#CHROM'] != 19) | (abs(combined_interact.POS - APOE_POS) > 0.5e6))][
    combined_cols
].reset_index(drop=True)

In [ ]:
# look for drops in significance
combined_interact[(combined_interact.P_female < gw_t) & (combined_interact.P_male > gw_t) & (combined_interact.P_female < combined_interact.P_initial) & ((combined_interact['#CHROM'] != 19) | (abs(combined_interact.POS - APOE_POS) > 0.5e6))][
    combined_cols
].reset_index(drop=True)

In [ ]:
# are there others
combined_interact[(combined_interact.islocmax_initial == 1) & (combined_interact['#CHROM'] == 11)][combined_cols]

In [ ]:
# Check male only < gw_t
combined_interact[(combined_interact.P_female > gw_t) & (combined_interact.P_male < gw_t) & ((combined_interact['#CHROM'] != 19) | (abs(combined_interact.POS - APOE_POS) > 0.5e6))][
    combined_cols
].sort_values('P_female').reset_index(drop=True)

In [ ]:
combined_interact[
    (combined_interact.P_female > gw_t) & (
        combined_interact.P_male < gw_t) & (
        combined_interact.P_male < combined_interact.P_initial) & (
        (combined_interact['#CHROM'] != 19) | (abs(combined_interact.POS - APOE_POS) > 0.5e6))][
    combined_cols
].reset_index(drop=True)

In [ ]:
combined_interact[combined_interact.ID == 'chr11:60177578:G:GT'][
    ['-log10_initial', '-log10_male', '-log10_female', '-log10_sex_interact', 'BETA_initial', 'BETA_male', 'BETA_female', 'BETA_sex_interact']
]

In [ ]:
 # are there any STRs nearby chr11:60177578 locus that are somewhat significant in the female analysis?
mask = (combined_interact['#CHROM'] == 11) & (abs(combined_interact.POS - 60177578) < 5e5) #& (combined_interact.P_female < 0.01)
combined_interact[mask][
    combined_cols
].sort_values('P_female').reset_index(drop=True).head()

In [ ]:
pos, range = 60177578, 0.25e6
fem_tmp = females_df[mask].copy()
fem_tmp['source'] = 'female samples'   
male_tmp = males_df[mask].copy()
male_tmp['source'] = 'male samples'
sns.scatterplot(
    pd.concat([fem_tmp, male_tmp]).reset_index(drop=True).sort_values('source', ascending=False), 
    hue='source', x='POS', y='-log10', palette={'female samples':'lightcoral', 'male samples':'cornflowerblue'}
)
hlines = plt.hlines(y=[-np.log10(gw_t), -np.log10(suggestive_t)], xmin=pos - range, xmax=pos + range, colors=['tab:red', 'k'], linestyles='dashed')

In [ ]:
sns.scatterplot(combined_interact[mask], y='-log10_female', x='POS')

In [ ]:
mask = (
    combined_interact.P_female < suggestive_t) | (
    combined_interact.P_male < suggestive_t) | (
    combined_interact.P_sex_interact < suggestive_t) | (
    combined_interact.P_initial < gw_t)
print(mask.sum())
combined_interact[mask][
    combined_cols
].sort_values(by=['#CHROM', 'POS']).reset_index(drop=True).to_csv(f'{results_path}/lead_strs_all_analyses.csv', sep='\t', index=False)